# 10 - run one pipeline stage

Generic runner. Set `STAGE` to the name of a script in `scripts/` (without `.py`); the notebook
installs the `msd` wheel, pulls `scripts/` and `data/` from the private HF workspace repo, runs
`python scripts/<STAGE>.py`, and pushes `results/<STAGE>/` back to the same repo.

Later tasks add the stage scripts (fine-tune, ReSta, evaluation, ...). This notebook does not
change when they do. Needs the `HF_TOKEN` secret, internet on and a GPU runtime. Setup:
`docs/runtime.md`.


In [ ]:
STAGE = "env_check"  # scripts/<STAGE>.py; later tasks add the real stage scripts
STAGE_ARGS = ""  # extra command-line arguments for the script, e.g. "--langs en,hi"
WORKSPACE_REPO = "YOUR-HF-USERNAME/msd-workspace"  # private HF dataset repo (docs/runtime.md)

In [ ]:
import os


def find_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:  # Kaggle: Add-ons > Secrets, label HF_TOKEN
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:  # Colab: Secrets (key icon), name HF_TOKEN, Notebook access on
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


token = find_token()
assert token, "HF_TOKEN not found: add it as a Kaggle or Colab secret (see docs/runtime.md)"
os.environ["HF_TOKEN"] = token  # scripts and msd read it from here; it is never printed
print("HF_TOKEN found")

In [ ]:
import subprocess
import sys

from huggingface_hub import HfApi, hf_hub_download

# Keep exactly one wheel in wheels/ of the workspace repo; the last one by name is installed.
files = HfApi().list_repo_files(WORKSPACE_REPO, repo_type="dataset", token=token)
wheel = sorted(f for f in files if f.startswith("wheels/") and f.endswith(".whl"))[-1]
path = hf_hub_download(WORKSPACE_REPO, wheel, repo_type="dataset", token=token)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", path], check=True)
print("installed", wheel)

from huggingface_hub import snapshot_download

snapshot_download(WORKSPACE_REPO, repo_type="dataset", local_dir=".", token=token,
                  allow_patterns=["scripts/*", "data/**"])

In [ ]:
import shlex
import subprocess
import sys

cmd = [sys.executable, f"scripts/{STAGE}.py", *shlex.split(STAGE_ARGS)]
print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

out = Path("results") / STAGE
if out.exists():
    HfApi().upload_folder(folder_path=str(out), path_in_repo=f"results/{STAGE}",
                          repo_id=WORKSPACE_REPO, repo_type="dataset", token=token)
    print("uploaded", out)
else:
    print("nothing to upload: no", out)